# 05. Field3 fine-tuning 실험 설계와 Colab 입력 패키지 준비

04 비교 결과를 바탕으로, 이번 노트북에서는 **학습을 바로 실행하지 않고 Colab에 올릴 입력 파일들을 정리**함.

왜 06 폴더를 확인하는가?

```text
이번 실험 후보 중 하나가 "기존 field1+2 fine-tuned 모델에서 field3로 추가 fine-tuning"이기 때문임.
따라서 10_experiments/06_lane_model_finetuning의 best.pth를 찾아야 함.
```

이번 단계의 출력:

```text
colab_inputs/field3_finetune_v2/
  field3_v1.tar.gz
  ResNet18_CULane.pth
  field12_finetune_v1_best.pth
  field12_finetune_v1_config.py
  experiment_plan.json
  README_upload_to_colab.md
```

이 패키지를 Google Drive에 올린 뒤, 다음 Colab 학습 노트북에서 A/B 실험을 실행함.


## 1. 경로와 후보 실험 정의

현재 가장 중요한 후보는 2개임.

```text
A. pretrained_to_field3
   ResNet18_CULane.pth -> field3_v1 fine-tuning

B. field12_best_to_field3
   field1+2 full fine-tuned best.pth -> field3_v1 추가 fine-tuning
```

A는 field3 데이터 자체의 효과를 보는 baseline이고, B는 이전 학습을 버리지 않고 실제 주행 데이터로 적응시키는 실전 후보임.


In [1]:
from pathlib import Path
import json
import os
import shutil
import tarfile
import time

PROJECT_ROOT = Path(r'~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization')
EXP11_DIR = PROJECT_ROOT / '10_experiments' / '11_clrkdnet_precision_tuning'
EXP06_DIR = PROJECT_ROOT / '10_experiments' / '06_lane_model_finetuning'

FIELD3_DATASET = EXP11_DIR / 'datasets' / 'field3_v1'
PRETRAINED_PTH = PROJECT_ROOT / '20_shared_assets' / 'models' / 'model' / 'ResNet18_CULane.pth'
FIELD12_FULL_RUN = EXP06_DIR / 'colab_outputs' / 'finetune_v1' / 'MapLane_Field1Field2_full' / '20260505_201647_lr_1e-04_b_8'
FIELD12_BEST_PTH = FIELD12_FULL_RUN / 'ckpt' / 'best.pth'
FIELD12_CONFIG = FIELD12_FULL_RUN / 'config.py'
FIELD12_LOG = FIELD12_FULL_RUN / 'log.txt'

PACK_DIR = EXP11_DIR / 'colab_inputs' / 'field3_finetune_v2'
PACK_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    'FIELD3_DATASET': FIELD3_DATASET,
    'PRETRAINED_PTH': PRETRAINED_PTH,
    'FIELD12_BEST_PTH': FIELD12_BEST_PTH,
    'FIELD12_CONFIG': FIELD12_CONFIG,
    'FIELD12_LOG': FIELD12_LOG,
    'PACK_DIR': PACK_DIR,
}
for name, path in paths.items():
    print(name, '->', path, 'exists=', path.exists(), 'size_MB=', round(path.stat().st_size/1024/1024, 2) if path.is_file() else None)

missing = [name for name, path in paths.items() if name != 'PACK_DIR' and not path.exists()]
assert not missing, missing


FIELD3_DATASET -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\11_clrkdnet_precision_tuning\datasets\field3_v1 exists= True size_MB= None
PRETRAINED_PTH -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\models\model\ResNet18_CULane.pth exists= True size_MB= 43.98
FIELD12_BEST_PTH -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8\ckpt\best.pth exists= True size_MB= 131.63
FIELD12_CONFIG -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8\config.py exists= True size_MB= 0.0
FIELD12_LOG -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_

## 2. Field3 dataset 상태 확인

Colab에 올리기 전에 `field3_v1`의 build/validation summary를 다시 확인함.


In [2]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

field3_summary = load_json(FIELD3_DATASET / 'build_summary.json')
field3_validation = load_json(FIELD3_DATASET / 'validation_summary.json')
print('build_summary')
print(json.dumps(field3_summary, indent=2, ensure_ascii=False))
print('\nvalidation_summary')
print(json.dumps(field3_validation, indent=2, ensure_ascii=False))
assert field3_summary['rows'] == 1679
assert field3_validation['issue_count'] == 0


build_summary
{
  "name": "field3_v1",
  "dataset_root": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\11_clrkdnet_precision_tuning\\datasets\\field3_v1",
  "source_dir": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\10_lane_autolabel_collection\\outputs\\20260501_191218_lane_autolabel_drive",
  "source_manifest": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\11_clrkdnet_precision_tuning\\review_outputs\\02_field3_label_quality_gate\\field3_train_candidate_manual.csv",
  "rows": 1679,
  "split_counts": {
    "train": 1378,
    "val": 164,
    "test": 137
  },
  "lane_count_manifest": {
    "1": 441,
    "2": 1238
  },
  "lane_count_lines": {
    "1": 441,
    "2": 1238
  },
  "geometry": {
    "raw_width": 1296,
    "raw_height": 972,
    "cut_height": 445,
    "model_width": 800,
    "model_height": 320,
    "max_lanes": 4,
    "seg_width": 16
  },
  "policy": 

## 3. 이전 field1+2 full fine-tuning 결과 확인

B 실험의 시작점은 `field12_finetune_v1_best.pth`임.

이 checkpoint는 이미 field1+2 pseudo dataset으로 12 epoch fine-tuning된 모델이고, field3로 이어 학습시켜 실제 주행 분포를 입히는 것이 목적임.


In [3]:
log_text = FIELD12_LOG.read_text(encoding='utf-8', errors='ignore')
metric_lines = [line for line in log_text.splitlines() if 'metric:' in line or 'Best metric:' in line or 'epoch:' in line]
print('metric/log tail:')
for line in metric_lines[-30:]:
    print(line)

print('\ncheckpoint size MB:', round(FIELD12_BEST_PTH.stat().st_size / 1024 / 1024, 2))
print('config size KB:', round(FIELD12_CONFIG.stat().st_size / 1024, 2))


metric/log tail:
2026-05-05 21:19:30,300 - clrkd.utils.recorder - INFO - epoch: 11  step: 10307  lr: 0.000001  loss: 1.7768  cls_loss: 0.5537  reg_xytl_loss: 0.3503  seg_loss: 0.0985  iou_loss: 0.7743  stage_0_acc: 98.6491  data: 0.0931  batch: 0.3283  eta: 0:03:21
2026-05-05 21:19:38,561 - clrkd.utils.recorder - INFO - epoch: 11  step: 10327  lr: 0.000001  loss: 1.9541  cls_loss: 0.6095  reg_xytl_loss: 0.4611  seg_loss: 0.0994  iou_loss: 0.7842  stage_0_acc: 98.6523  data: 0.0840  batch: 0.4123  eta: 0:03:14
2026-05-05 21:19:45,415 - clrkd.utils.recorder - INFO - epoch: 11  step: 10347  lr: 0.000001  loss: 1.9188  cls_loss: 0.6326  reg_xytl_loss: 0.4042  seg_loss: 0.1015  iou_loss: 0.7805  stage_0_acc: 98.4017  data: 0.0896  batch: 0.3431  eta: 0:03:07
2026-05-05 21:19:53,439 - clrkd.utils.recorder - INFO - epoch: 11  step: 10367  lr: 0.000001  loss: 1.7030  cls_loss: 0.5742  reg_xytl_loss: 0.3336  seg_loss: 0.0989  iou_loss: 0.6962  stage_0_acc: 98.4928  data: 0.0811  batch: 0.4022  

## 4. Colab 입력 archive 생성

`field3_v1` 전체를 `field3_v1.tar.gz`로 묶음.

주의:

```text
이 archive에는 image, .lines.txt, laneseg_label_w16, list, build_summary가 포함됨.
_review_overlays는 없음.
```


In [4]:
FIELD3_TAR = PACK_DIR / 'field3_v1.tar.gz'

REBUILD_TAR = True
if FIELD3_TAR.exists() and REBUILD_TAR:
    FIELD3_TAR.unlink()

if not FIELD3_TAR.exists():
    start = time.time()
    with tarfile.open(FIELD3_TAR, 'w:gz') as tf:
        tf.add(FIELD3_DATASET, arcname=FIELD3_DATASET.name)
    elapsed = time.time() - start
    print('created:', FIELD3_TAR)
    print('size MB:', round(FIELD3_TAR.stat().st_size / 1024 / 1024, 2), 'elapsed sec:', round(elapsed, 1))
else:
    print('already exists:', FIELD3_TAR, 'size MB:', round(FIELD3_TAR.stat().st_size / 1024 / 1024, 2))


created: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\11_clrkdnet_precision_tuning\colab_inputs\field3_finetune_v2\field3_v1.tar.gz
size MB: 311.66 elapsed sec: 51.2


## 5. Checkpoint와 config 복사

Colab 폴더 하나만 업로드하면 되도록 필요한 모델 파일을 패키지 폴더에 복사함.


In [5]:
copy_plan = [
    (PRETRAINED_PTH, PACK_DIR / 'ResNet18_CULane.pth'),
    (FIELD12_BEST_PTH, PACK_DIR / 'field12_finetune_v1_best.pth'),
    (FIELD12_CONFIG, PACK_DIR / 'field12_finetune_v1_config.py'),
    (FIELD12_LOG, PACK_DIR / 'field12_finetune_v1_log.txt'),
]

for src, dst in copy_plan:
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
        print('copied:', dst.name, 'MB=', round(dst.stat().st_size / 1024 / 1024, 2))
    else:
        print('exists:', dst.name, 'MB=', round(dst.stat().st_size / 1024 / 1024, 2))


copied: ResNet18_CULane.pth MB= 43.98
copied: field12_finetune_v1_best.pth MB= 131.63
copied: field12_finetune_v1_config.py MB= 0.0
copied: field12_finetune_v1_log.txt MB= 0.15


## 6. 실험 계획 JSON 작성

다음 Colab 학습 노트북은 이 JSON을 기준으로 어떤 실험을 돌릴지 알 수 있음.


In [6]:
experiment_plan = {
    'name': 'field3_finetune_v2',
    'created_from': str(EXP11_DIR),
    'dataset': {
        'name': 'field3_v1',
        'archive': 'field3_v1.tar.gz',
        'rows': field3_summary['rows'],
        'split_counts': field3_summary['split_counts'],
        'lane_count': field3_summary['lane_count_lines'],
        'geometry': field3_summary['geometry'],
    },
    'base_checkpoints': {
        'pretrained_resnet18_culane': 'ResNet18_CULane.pth',
        'field12_finetune_v1_best': 'field12_finetune_v1_best.pth',
    },
    'experiments': [
        {
            'id': 'A_pretrained_to_field3_smoke_then_full',
            'base_checkpoint': 'ResNet18_CULane.pth',
            'purpose': 'field3_v1 데이터 자체의 supervised fine-tuning 효과 확인',
            'lr': 1e-4,
            'smoke_epochs': 1,
            'full_epochs': 12,
            'batch_size': 8,
        },
        {
            'id': 'B_field12_best_to_field3_smoke_then_full',
            'base_checkpoint': 'field12_finetune_v1_best.pth',
            'purpose': 'field1+2로 적응된 모델에 실제 주행 field3 분포를 추가 적응',
            'lr': 5e-5,
            'smoke_epochs': 1,
            'full_epochs': 8,
            'batch_size': 8,
        },
    ],
    'fixed_model_config': {
        'sample_y': 'range(971, 444, -20)',
        'ori_img_w': 1296,
        'ori_img_h': 972,
        'img_w': 800,
        'img_h': 320,
        'cut_height': 445,
        'dataset_path': './data/field3_v1',
        'diff_path': None,
        'test_parameters': {'conf_threshold': 0.35, 'nms_thres': 50, 'nms_topk': 4},
    },
    'decision_after_training': [
        'field3-only가 pretrained보다 라인 출력이 살아나는지 확인',
        'field12_best_to_field3가 이전 v1보다 실제 주행 이미지에서 개선되는지 확인',
        '둘 다 불안정하면 CLRKDNet을 주행 핵심 모델로 계속 가져갈지 재판단',
    ],
}
plan_path = PACK_DIR / 'experiment_plan.json'
plan_path.write_text(json.dumps(experiment_plan, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(experiment_plan, indent=2, ensure_ascii=False))


{
  "name": "field3_finetune_v2",
  "created_from": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\11_clrkdnet_precision_tuning",
  "dataset": {
    "name": "field3_v1",
    "archive": "field3_v1.tar.gz",
    "rows": 1679,
    "split_counts": {
      "train": 1378,
      "val": 164,
      "test": 137
    },
    "lane_count": {
      "1": 441,
      "2": 1238
    },
    "geometry": {
      "raw_width": 1296,
      "raw_height": 972,
      "cut_height": 445,
      "model_width": 800,
      "model_height": 320,
      "max_lanes": 4,
      "seg_width": 16
    }
  },
  "base_checkpoints": {
    "pretrained_resnet18_culane": "ResNet18_CULane.pth",
    "field12_finetune_v1_best": "field12_finetune_v1_best.pth"
  },
  "experiments": [
    {
      "id": "A_pretrained_to_field3_smoke_then_full",
      "base_checkpoint": "ResNet18_CULane.pth",
      "purpose": "field3_v1 데이터 자체의 supervised fine-tuning 효과 확인",
      "lr": 0.0001,
      "smoke_epochs": 1,
  

## 7. Colab 업로드 README 생성

실제로 Google Drive에 올릴 파일과 다음 작업을 사람 눈으로 확인하기 위한 안내문을 생성함.


In [7]:
readme = f"""# field3_finetune_v2 Colab input package

이 폴더는 CLRKDNet field3 fine-tuning v2 입력 패키지임.

업로드 위치 예시:

```text
MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/04_CLRKDNet_Field3_Finetune_v2/
```

필수 파일:

```text
field3_v1.tar.gz
ResNet18_CULane.pth
field12_finetune_v1_best.pth
experiment_plan.json
```

실험 후보:

```text
A. ResNet18_CULane.pth -> field3_v1 fine-tuning
B. field12_finetune_v1_best.pth -> field3_v1 추가 fine-tuning
```

현재 패키지 생성 위치:

```text
{PACK_DIR}
```
"""
readme_path = PACK_DIR / 'README_upload_to_colab.md'
readme_path.write_text(readme, encoding='utf-8')
print(readme_path)
print(readme)


~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\11_clrkdnet_precision_tuning\colab_inputs\field3_finetune_v2\README_upload_to_colab.md
# field3_finetune_v2 Colab input package

이 폴더는 CLRKDNet field3 fine-tuning v2 입력 패키지임.

업로드 위치 예시:

```text
MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/04_CLRKDNet_Field3_Finetune_v2/
```

필수 파일:

```text
field3_v1.tar.gz
ResNet18_CULane.pth
field12_finetune_v1_best.pth
experiment_plan.json
```

실험 후보:

```text
A. ResNet18_CULane.pth -> field3_v1 fine-tuning
B. field12_finetune_v1_best.pth -> field3_v1 추가 fine-tuning
```

현재 패키지 생성 위치:

```text
~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\11_clrkdnet_precision_tuning\colab_inputs\field3_finetune_v2
```



## 8. 패키지 최종 검증

이 셀에서 나온 폴더 안 파일들을 Google Drive로 올리면 됨.


In [8]:
required_names = [
    'field3_v1.tar.gz',
    'ResNet18_CULane.pth',
    'field12_finetune_v1_best.pth',
    'field12_finetune_v1_config.py',
    'experiment_plan.json',
    'README_upload_to_colab.md',
]
rows = []
for name in required_names:
    p = PACK_DIR / name
    rows.append({
        'name': name,
        'exists': p.exists(),
        'size_MB': round(p.stat().st_size / 1024 / 1024, 2) if p.exists() else None,
        'path': str(p),
    })
import pandas as pd
package_df = pd.DataFrame(rows)
display(package_df)
assert package_df['exists'].all(), package_df
print('PACK_DIR:', PACK_DIR)
print('total size MB:', round(sum((PACK_DIR / name).stat().st_size for name in required_names if (PACK_DIR / name).exists()) / 1024 / 1024, 2))


,name,exists,size_MB,path
0,field3_v1.tar.gz,True,311.66,~\02_Projects\University\26-1_Em...
1,ResNet18_CULane.pth,True,43.98,~\02_Projects\University\26-1_Em...
2,field12_finetune_v1_best.pth,True,131.63,~\02_Projects\University\26-1_Em...
3,field12_finetune_v1_config.py,True,0.00,~\02_Projects\University\26-1_Em...
4,experiment_plan.json,True,0.00,~\02_Projects\University\26-1_Em...
5,README_upload_to_colab.md,True,0.00,~\02_Projects\University\26-1_Em...


PACK_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\11_clrkdnet_precision_tuning\colab_inputs\field3_finetune_v2
total size MB: 487.28


## 9. 실행 후 메모

아직 실행 전임.

실행 후 확인할 것:

```text
field3_v1.tar.gz가 정상 생성됐는가
ResNet18_CULane.pth와 field12_finetune_v1_best.pth가 같은 폴더에 복사됐는가
experiment_plan.json의 실험 A/B가 맞는가
Google Drive에 이 폴더를 통째로 업로드할 준비가 됐는가
```

다음 노트북은 Colab에서 실제 학습을 실행하는 `06_field3_finetune_v2_colab.ipynb`가 됨.
